# 01 — Load Relational Parquet Tables

This notebook validates that the Medical Insight Explorer Agent can load the cleaned Parquet outputs generated by the upstream Healthcare-Data-Cleaning project.

The goal is not to build the agent yet. The goal is to confirm that all required relational healthcare tables are available and readable.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agent.data_loader import HealthcareDataLoader

In [2]:
loader = HealthcareDataLoader(data_dir=PROJECT_ROOT / "data" / "processed")
tables = loader.load_tables()

list(tables.keys())

['train_beneficiary',
 'test_beneficiary',
 'train_inpatient',
 'test_inpatient',
 'train_outpatient',
 'test_outpatient',
 'train_labels',
 'test_labels']

In [3]:
shape_summary = loader.get_table_shapes()
shape_summary

,table_name,rows,columns
0,test_beneficiary,63968,26
1,test_inpatient,9551,30
2,test_labels,1353,1
3,test_outpatient,125841,27
4,train_beneficiary,138556,26
5,train_inpatient,40474,30
6,train_labels,5410,2
7,train_outpatient,517737,27


In [4]:
for table_name, df in tables.items():
    print(f"\n--- {table_name} ---")
    print(df.shape)
    print(df.columns.tolist()[:15])


--- train_beneficiary ---
(138556, 26)
['BeneID', 'DOB', 'DOD', 'Gender', 'Race', 'RenalDiseaseIndicator', 'State', 'County', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary']

--- test_beneficiary ---
(63968, 26)
['BeneID', 'DOB', 'DOD', 'Gender', 'Race', 'RenalDiseaseIndicator', 'State', 'County', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary']

--- train_inpatient ---
(40474, 30)
['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode', 'DeductibleAmtPaid', 'DischargeDt', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1']

--- test_inpatient ---
(9551, 30)
['BeneID', 'ClaimID', 'ClaimStartDt', 'Claim

In [5]:
required_key_checks = {
    "train_beneficiary": ["BeneID"],
    "test_beneficiary": ["BeneID"],
    "train_inpatient": ["BeneID", "ClaimID"],
    "test_inpatient": ["BeneID", "ClaimID"],
    "train_outpatient": ["BeneID", "ClaimID"],
    "test_outpatient": ["BeneID", "ClaimID"],
    "train_labels": ["Provider"],
    "test_labels": ["Provider"],
}

for table_name, required_columns in required_key_checks.items():
    df = tables[table_name]

    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"{table_name} is missing columns: {missing_columns}")

print("All required key columns are present.")

All required key columns are present.


In [6]:
train_beneficiary_ids = set(tables["train_beneficiary"]["BeneID"])
train_inpatient_ids = set(tables["train_inpatient"]["BeneID"])
train_outpatient_ids = set(tables["train_outpatient"]["BeneID"])

missing_inpatient_beneids = train_inpatient_ids - train_beneficiary_ids
missing_outpatient_beneids = train_outpatient_ids - train_beneficiary_ids

print(f"Missing train inpatient BeneIDs: {len(missing_inpatient_beneids)}")
print(f"Missing train outpatient BeneIDs: {len(missing_outpatient_beneids)}")

Missing train inpatient BeneIDs: 0
Missing train outpatient BeneIDs: 0


## Result

All required cleaned Parquet tables were loaded successfully.

This confirms that the Medical Insight Explorer Agent can consume the validated relational outputs from the upstream Healthcare-Data-Cleaning project.